In [0]:
import requests
import pandas as pd
from pyspark.sql import functions as F

In [0]:
# --- Configuration ---
CATALOG = "iran_israel_capstone_project"
SCHEMA = "bronze"
LANDING_BASE_PATH = "abfss://capstonecontainer@iranisrael65.dfs.core.windows.net/landing_zone/macro_data"

In [0]:
API_KEY = "YOURC3ODNMPQM5TL20OM"  
BASE_URL = "https://www.alphavantage.co/query"

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

In [0]:
def fetch_and_promote_macro_data(function, name, interval=None):
    # --- 1. API to ADLS Landing Zone ---
    params = {"function": function, "apikey": API_KEY}
    if interval: params["interval"] = interval
    
    print(f"Fetching {name} from Alpha Vantage...")
    response = requests.get(BASE_URL, params=params)
    data = response.json()
    
    if "data" in data:
        df_pd = pd.DataFrame(data["data"])
        spark_df = spark.createDataFrame(df_pd)
        
        target_landing_path = f"{LANDING_BASE_PATH}/{name.lower()}"
        spark_df.write.mode("overwrite").parquet(target_landing_path)
        print(f"Landed {name} in {target_landing_path}")
        
        # --- 2. ADLS Landing Zone to Unity Catalog (Bronze) ---
        df_raw = spark.read.parquet(target_landing_path)
        
        df_bronze = df_raw.withColumnRenamed("date", "record_date") \
                          .withColumn("record_date", F.to_date("record_date")) \
                          .withColumn("source_file", F.lit(f"alpha_vantage_{function}")) \
                          .withColumn("ingestion_timestamp", F.current_timestamp())
        
        table_name = f"{name.lower()}_raw"
        df_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
        print(f"Promoted to bronze.{table_name}")
    else:
        print(f"Error fetching {name}: {data.get('Information', 'Unknown Error')}")

In [0]:
# Execute
fetch_and_promote_macro_data("CPI", "macro_cpi", interval="monthly")
fetch_and_promote_macro_data("WTI", "macro_wti", interval="daily")
fetch_and_promote_macro_data("BRENT", "macro_brent_alpha", interval="daily")